# 03 · Detección de objetos: de IoU a YOLO11 y DETR 

El bloque más preguntado. Al terminar:
- Implementar **IoU** y **NMS** en una pizarra (live coding típico).
- Explicar **precision, recall, AP, mAP@50 y mAP@50:95** y cómo se calculan.
- Explicar **cómo funciona YOLO por dentro** (backbone/neck/head, anchor-free, qué sale del modelo) y qué cambió
  entre versiones.
- Usar **Ultralytics** con soltura: predict, val, train, parámetros clave.
- Explicar **DETR / RT-DETR / RF-DETR** y por qué no necesitan NMS.
- Hacer **análisis de errores** de un detector.

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
from pathlib import Path
import time
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

cap = cv2.VideoCapture(str(DATA / "people.mp4"))
ok, frame = cap.read()
cap.release()

## 1. La tarea y los formatos de caja

| Tarea | Salida |
|---|---|
| Clasificación | una clase por imagen |
| **Detección** | lista de (caja, clase, confianza) |
| Segmentación semántica | clase por píxel (sin distinguir instancias) |
| Segmentación de instancias | máscara por objeto |
| Pose / keypoints | caja + K puntos (x, y, visibilidad) por objeto |

Formatos de caja (confundirlos = bug clásico):
- **xyxy**: `(x1, y1, x2, y2)` esquinas — Pascal VOC, torchvision, salida de Ultralytics `boxes.xyxy`.
- **xywh**: `(x1, y1, w, h)` esquina superior-izquierda + tamaño — **COCO**, formato **MOT**.
- **cxcywh**: centro + tamaño — salida cruda de YOLO y DETR.
- **YOLO txt**: `clase cx cy w h` **normalizados a [0, 1]** por el ancho/alto de la imagen.

In [ ]:
def xywh2xyxy(b):
    b = np.asarray(b, dtype=np.float32).copy()
    b[..., 2] += b[..., 0]; b[..., 3] += b[..., 1]
    return b

def cxcywh2xyxy(b):
    b = np.asarray(b, dtype=np.float32)
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=-1)

print(xywh2xyxy([10, 20, 30, 40]), cxcywh2xyxy([25, 40, 30, 40]))

## 2. ✍️ EJERCICIO: IoU vectorizado

$$ IoU(A,B) = \frac{|A \cap B|}{|A \cup B|} = \frac{inter}{area_A + area_B - inter} $$

La intersección de dos cajas xyxy es otra caja: `x1 = max(x1a, x1b)`, `x2 = min(x2a, x2b)` (igual en y), y su
área es `max(0, x2-x1) * max(0, y2-y1)` (el `max(0, ·)` cubre el caso sin solape).

Implementa `box_iou(a, b)` con `a: (N,4)`, `b: (M,4)` → matriz `(N, M)` **sin bucles** (broadcasting:
`a[:, None, :]` contra `b[None, :, :]`). Esta matriz es justo la que usará el tracker para asociar.

In [ ]:
# SOLUCIÓN
def box_iou(a, b):
    a, b = np.asarray(a, np.float32), np.asarray(b, np.float32)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])   # (N, M, 2) esquina sup-izq de la intersección
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])   # (N, M, 2) esquina inf-der
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

In [ ]:
from torchvision.ops import box_iou as tv_iou
A = np.array([[0, 0, 10, 10], [5, 5, 15, 15]], np.float32)
B = np.array([[0, 0, 10, 10], [10, 10, 20, 20], [100, 100, 110, 110]], np.float32)
print(np.round(box_iou(A, B), 3))
r = np.random.rand(50, 2) * 150; R = np.hstack([r, r + np.random.rand(50, 2) * 100]).astype(np.float32)
assert np.allclose(box_iou(R, R[:20]), tv_iou(torch.tensor(R), torch.tensor(R[:20])).numpy(), atol=1e-5)
print("✅ coincide con torchvision")

**Variantes de IoU** (se usan como pérdida de regresión de cajas):
- **GIoU**: resta la fracción del rectángulo envolvente que no cubren → tiene gradiente aunque no se solapen.
- **DIoU**: penaliza la distancia entre centros. **CIoU** (YOLOv8/11): DIoU + consistencia de aspecto.

## 3. ✍️ EJERCICIO: Non-Maximum Suppression

Un detector produce **miles de cajas candidatas** (YOLO11 a 640: 8400) y muchas cubren el mismo objeto.
NMS greedy:
1. Ordena por confianza descendente.
2. Coge la mejor, guárdala, y **elimina todas las restantes con IoU > umbral** con ella.
3. Repite con las que quedan.

**NMS por clase**: solo se suprimen cajas de la misma clase. Truco eficiente (el que usan torchvision y
Ultralytics): desplazar cada caja `clase * offset_grande` para que cajas de clases distintas nunca solapen y
hacer un único NMS.

Implementa `nms(boxes, scores, iou_thr)` → índices conservados.

In [ ]:
# SOLUCIÓN
def nms(boxes, scores, iou_thr=0.5):
    order = np.argsort(-scores)
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        ious = box_iou(boxes[i:i + 1], boxes[order[1:]])[0]
        order = order[1:][ious <= iou_thr]
    return np.array(keep, dtype=int)

def batched_nms(boxes, scores, classes, iou_thr=0.5):
    offset = classes[:, None] * (boxes.max() + 1)
    return nms(boxes + offset, scores, iou_thr)

In [ ]:
from torchvision.ops import nms as tv_nms
S = np.random.rand(50).astype(np.float32)
mine = nms(R, S, 0.5)
ref = tv_nms(torch.tensor(R), torch.tensor(S), 0.5).numpy()
assert set(mine) == set(ref)
print(f"✅ NMS correcto: {len(R)} → {len(mine)} cajas")

**Limitaciones de NMS** (pregunta frecuente, y muy relevante en multitudes/deporte):
- Dos personas muy solapadas (abrazo, melé, jugadores en un córner) → NMS puede **borrar una real** (FN).
  Subir el umbral IoU → más duplicados (FP). Es un compromiso.
- Alternativas: **Soft-NMS** (baja la confianza en vez de borrar), umbrales adaptativos, o **modelos sin NMS**
  (DETR, YOLOv10, YOLO26) que aprenden a no producir duplicados.
- Coste: secuencial y depende del número de candidatos; en GPU se usa versión batched. Un `conf` mínimo
  razonable reduce mucho el trabajo de NMS.

## 4. YOLO por dentro

### Arquitectura (YOLOv8 / YOLO11)
```
imagen 640×640 ─► BACKBONE (CSP: C2f en v8, C3k2 + C2PSA en v11)
                    │ P3 (80×80, stride 8)   P4 (40×40, stride 16)   P5 (20×20, stride 32)
                    ▼
                  NECK (FPN + PAN: mezcla info de arriba-abajo y abajo-arriba entre escalas)
                    ▼
                  HEAD desacoplada por escala: rama de cajas (4×reg_max, DFL) + rama de clases (80, sigmoide)
```
- **Una predicción por celda de cada mapa**: 80² + 40² + 20² = **8400** candidatos a 640.
- **Anchor-free** (desde v8): cada celda predice directamente distancias a los 4 lados de la caja (l, t, r, b),
  en vez de ajustar *anchor boxes* predefinidas (v3–v5). Menos hiperparámetros, generaliza mejor a formas raras.
- **Head desacoplada**: ramas separadas para clasificación y regresión (aprenden cosas distintas).
- **Sin objectness** (v8+): la confianza es directamente la probabilidad de clase.
- **Asignación de etiquetas: Task-Aligned Assigner** (TAL): en entrenamiento, qué predicciones son "positivas"
  para cada objeto se decide según una mezcla de score de clase e IoU.
- **Pérdidas**: BCE (clases) + **CIoU** + **DFL** (cajas).
- Escalas n/s/m/l/x: mismo diseño, distinta anchura/profundidad (n ≈ 2.6M params, x ≈ 57M).

### Evolución (una frase por versión)
| Versión | Novedad clave |
|---|---|
| YOLOv3 | Darknet-53, 3 escalas, anchors |
| YOLOv5 (Ultralytics) | PyTorch, ingeniería y augmentations (mosaic), anchors + objectness |
| **YOLOv8** | **anchor-free**, head desacoplada, C2f, DFL, multitarea (det/seg/pose/obb/cls) |
| YOLOv10 | **NMS-free** por *dual label assignment* (uno-a-muchos en entrenamiento, uno-a-uno en inferencia) |
| **YOLO11** | C3k2 + atención C2PSA; más preciso con menos parámetros que v8 |
| YOLO26 | end-to-end sin NMS y sin DFL, pensado para edge/exportación |

### Veámoslo: salida cruda del modelo

In [ ]:
model = YOLO(str(DATA / "yolo11n.pt"))
model.info()
net = model.model.eval()
x = torch.zeros(1, 3, 640, 640)
with torch.no_grad():
    raw = net(x)
raw = raw[0] if isinstance(raw, (list, tuple)) else raw
print("salida cruda:", tuple(raw.shape), "= (batch, 4 caja + 80 clases, 8400 candidatos)")
print("strides:", net.stride.tolist())

### ✍️ EJERCICIO: decodificar la salida cruda tú mismo

Reproduce el postproceso de Ultralytics con lo que has implementado:
1. `letterbox` del frame a 640 (del notebook 01), BGR→RGB, `/255`, HWC→CHW, batch.
2. `raw = net(x)` → `(1, 84, 8400)`; transponer a `(8400, 84)`.
3. Las 4 primeras columnas son `cx, cy, w, h` en píxeles del 640×640; las 80 siguientes, scores de clase
   (ya con sigmoide).
4. Score = max por fila, clase = argmax; filtrar `score > 0.25`.
5. `cxcywh → xyxy`, NMS por clase, deshacer el letterbox.

Compara el número de detecciones con `model(frame)`.

In [ ]:
def letterbox(img, size=640, color=(114, 114, 114)):
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nw, nh = round(w * r), round(h * r)
    out = np.full((size, size, 3), color, dtype=img.dtype)
    dx, dy = (size - nw) // 2, (size - nh) // 2
    out[dy:dy + nh, dx:dx + nw] = cv2.resize(img, (nw, nh))
    return out, r, (dx, dy)

In [ ]:
# SOLUCIÓN
def my_yolo_detect(frame, conf=0.25, iou=0.7):
    lb, r, (dx, dy) = letterbox(frame)
    x = torch.from_numpy(lb[:, :, ::-1].copy()).permute(2, 0, 1).float().div(255).unsqueeze(0)
    with torch.no_grad():
        raw = net(x)
    raw = raw[0] if isinstance(raw, (list, tuple)) else raw
    p = raw[0].T.numpy()                       # (8400, 84)
    cls_scores = p[:, 4:]
    scores, classes = cls_scores.max(1), cls_scores.argmax(1)
    m = scores > conf
    boxes = cxcywh2xyxy(p[m, :4]); scores, classes = scores[m], classes[m]
    keep = batched_nms(boxes, scores, classes, iou)
    boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - dx) / r
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - dy) / r
    return boxes, scores, classes

In [ ]:
b, s, c = my_yolo_detect(frame)
ref = model(frame, conf=0.25, iou=0.7, device="cpu", verbose=False)[0]
print(f"mi decodificación: {len(b)} detecciones | Ultralytics: {len(ref.boxes)}")
if len(ref.boxes):
    m = box_iou(ref.boxes.xyxy.cpu().numpy(), b).max(1)
    print(f"{(m > 0.5).sum()}/{len(m)} detecciones de Ultralytics tienen pareja en las mías (IoU media {m.mean():.2f})")
# (pequeñas diferencias vienen del padding: Ultralytics hace letterbox "mínimo" múltiplo de 32 en predict)

## 5. Ultralytics: lo que tienes que saber usar

```python
model = YOLO("yolo11n.pt")                 # det | "-seg" | "-pose" | "-obb" | "-cls"
results = model(source, conf=0.25, iou=0.7, imgsz=640, classes=[0], device="mps", half=False, stream=True)
for r in results:                           # stream=True → generador, no carga todo el vídeo en RAM
    r.boxes.xyxy, r.boxes.conf, r.boxes.cls, r.boxes.id   # (id solo con model.track)
    r.masks, r.keypoints, r.speed, r.plot()
model.val(data="coco8.yaml")                # métricas
model.train(data="data.yaml", epochs=100, imgsz=640, batch=16)
model.export(format="onnx")                 # onnx, engine (TensorRT), coreml, openvino, tflite...
model.track(source, tracker="bytetrack.yaml", persist=True)   # notebook 06
```

Parámetros que importan en producción:
- `conf`: umbral de confianza. Bajo → más recall, más FP. **Para tracking se suele bajar** (0.1) porque
  ByteTrack aprovecha las detecciones de baja confianza.
- `iou`: umbral de NMS. `imgsz`: resolución (más grande → objetos pequeños mejor, más lento).
- `classes=[0]`: solo personas. `half=True`: FP16 en GPU. `batch`: varias imágenes a la vez.

In [ ]:
res = model(frame, conf=0.25, classes=[0], device=DEVICE, verbose=False)[0]
print(len(res.boxes), "personas | speed ms:", {k: round(v, 1) for k, v in res.speed.items()})
print(res.boxes.xyxy[:3].cpu().numpy().round(1), res.boxes.conf[:3].cpu().numpy().round(2))
plt.figure(figsize=(12, 7)); plt.imshow(res.plot()[:, :, ::-1]); plt.axis("off");

In [ ]:
# Velocidad según tamaño de entrada (warm-up primero: la 1ª inferencia siempre es lenta)
for sz in [320, 640, 960, 1280]:
    model(frame, imgsz=sz, device=DEVICE, verbose=False)
    t0 = time.perf_counter()
    for _ in range(10):
        r = model(frame, imgsz=sz, classes=[0], device=DEVICE, verbose=False)[0]
    print(f"imgsz={sz:5d}: {1000*(time.perf_counter()-t0)/10:6.1f} ms/frame, {len(r.boxes):3d} personas")

Observa: a más resolución, **más personas detectadas** (las lejanas son pequeñas) pero más lento.
Este compromiso precisión/latencia es *el* tema de conversación en un puesto con "tiempos de procesamiento
reducidos".

## 6. Métricas: precision, recall, AP, mAP

**Emparejamiento**: una predicción es **TP** si tiene IoU ≥ umbral (0.5) con un GT de su clase **no
emparejado ya** (cada GT solo vale para una predicción, recorriendo las predicciones de mayor a menor confianza).
Si no, **FP**. Los GT sin emparejar son **FN**.

- **Precision** = TP / (TP + FP) → "de lo que digo que es persona, cuánto lo es".
- **Recall** = TP / (TP + FN) → "de las personas que hay, cuántas encuentro".
- Al barrer el umbral de confianza de alto a bajo obtienes la **curva PR**.
- **AP** = área bajo la curva PR (con la precisión "interpolada": la precisión en recall r es la máxima
  precisión para cualquier recall ≥ r). COCO la muestrea en 101 puntos de recall.
- **mAP@50** = media de AP sobre clases con IoU 0.5. **mAP@50:95** = media además sobre IoU 0.50, 0.55, …, 0.95
  → premia **localización precisa**. Es la métrica principal de COCO (la que reporta Ultralytics como `mAP50-95`).
- **F1** = 2PR/(P+R): útil para **elegir el umbral de confianza** de despliegue (el máximo de la curva F1).

✍️ **EJERCICIO**: implementa `average_precision(preds, gts, iou_thr)` para una clase.
`preds`: lista de `(img_id, score, box)`; `gts`: dict `img_id → array (K,4)`.

In [ ]:
# SOLUCIÓN
def average_precision(preds, gts, iou_thr=0.5):
    n_gt = sum(len(g) for g in gts.values())
    preds = sorted(preds, key=lambda p: -p[1])
    used = {k: np.zeros(len(g), bool) for k, g in gts.items()}
    tp = np.zeros(len(preds)); fp = np.zeros(len(preds))
    for i, (img, score, box) in enumerate(preds):
        g = gts.get(img, np.zeros((0, 4)))
        if len(g):
            ious = box_iou(np.array([box]), g)[0]
            j = ious.argmax()
            if ious[j] >= iou_thr and not used[img][j]:
                tp[i] = 1; used[img][j] = True; continue
        fp[i] = 1
    tp_c, fp_c = np.cumsum(tp), np.cumsum(fp)
    recall = tp_c / max(n_gt, 1)
    precision = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    # interpolación COCO: precisión monótona decreciente, 101 puntos de recall
    prec_interp = np.maximum.accumulate(precision[::-1])[::-1]
    rs = np.linspace(0, 1, 101)
    idx = np.searchsorted(recall, rs, side="left")
    ap = np.mean([prec_interp[i] if i < len(prec_interp) else 0 for i in idx])
    return ap, precision, recall

Para probarlo sin dataset anotado, usamos un truco muy útil en la práctica: **pseudo-GT** con un modelo grande.
Tomamos las detecciones de `yolo11x` (si tienes tiempo de descargarlo, ~110 MB; si no, `yolo11s`) a alta
resolución como "verdad" y evaluamos `yolo11n` a 640 contra ellas. (Ojo: es una aproximación — en el trabajo
real se evalúa contra anotaciones humanas.)

In [ ]:
teacher = YOLO(str(DATA / "yolo11s.pt"))
cap = cv2.VideoCapture(str(DATA / "people.mp4"))
frames = []
for i in range(300):
    ok, f = cap.read()
    if not ok: break
    if i % 15 == 0: frames.append(f)
cap.release()

gts, preds = {}, []
for k, f in enumerate(frames):
    t = teacher(f, imgsz=1280, conf=0.4, classes=[0], device=DEVICE, verbose=False)[0]
    gts[k] = t.boxes.xyxy.cpu().numpy()
    s = model(f, imgsz=640, conf=0.01, classes=[0], device=DEVICE, verbose=False)[0]
    for bx, sc in zip(s.boxes.xyxy.cpu().numpy(), s.boxes.conf.cpu().numpy()):
        preds.append((k, float(sc), bx))

ap50, P, Rc = average_precision(preds, gts, 0.5)
aps = [average_precision(preds, gts, t)[0] for t in np.arange(0.5, 0.96, 0.05)]
print(f"AP@50 = {ap50:.3f} | AP@50:95 = {np.mean(aps):.3f}")
f1 = 2 * P * Rc / (P + Rc + 1e-9)
best = f1.argmax()
print(f"mejor F1 = {f1[best]:.3f} con conf ≈ {sorted([p[1] for p in preds], reverse=True)[best]:.2f}")
plt.plot(Rc, P); plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Curva PR yolo11n vs pseudo-GT"); plt.grid();

## 7. Fine-tuning con Ultralytics

Formato del dataset (lo verás a fondo en el notebook 05):
```
dataset/
  images/train/*.jpg   images/val/*.jpg
  labels/train/*.txt   labels/val/*.txt      ← una línea por objeto: "cls cx cy w h" normalizado
data.yaml:  path: dataset   train: images/train   val: images/val   names: {0: person, 1: ball}
```

Entrenamos sobre `coco128` (128 imágenes de COCO, se descarga sola ~7 MB) solo para ver el flujo. En
el Mac con MPS tarda unos minutos. **Lo importante es saber leer las salidas**:
- `runs/detect/<nombre>/results.csv` → loss de box/cls/dfl en train y val, P, R, mAP por época.
- `results.png`, `confusion_matrix.png`, `BoxPR_curve.png`, `BoxF1_curve.png`, `val_batch*_pred.jpg`.
- `weights/best.pt` (mejor mAP en val) y `last.pt` (para reanudar con `resume=True`).

Hiperparámetros que deberías saber justificar: `epochs`, `imgsz`, `batch`, `lr0`, `optimizer`, `patience`
(early stopping), `freeze=10` (congela las primeras 10 capas), `mosaic`, `close_mosaic` (desactiva mosaic las
últimas épocas), `hsv_h/s/v`, `fliplr`, `scale`, `degrees`, `cache=True`.

In [ ]:
ft = YOLO(str(DATA / "yolo11n.pt"))
train_res = ft.train(data="coco128.yaml", epochs=3, imgsz=640, batch=16, device=DEVICE,
                     project=str(DATA / "runs"), name="coco128_ft", exist_ok=True,
                     plots=True, verbose=False, workers=0)

In [ ]:
metrics = ft.val(data="coco128.yaml", device=DEVICE, plots=False, verbose=False,
                 project=str(DATA / "runs"), name="coco128_val", exist_ok=True)
print(f"mAP50-95 = {metrics.box.map:.3f} | mAP50 = {metrics.box.map50:.3f} | "
      f"P = {metrics.box.mp:.3f} | R = {metrics.box.mr:.3f}")
import pandas as pd
pd.read_csv(DATA / "runs/coco128_ft/results.csv").round(3)

## 8. La familia DETR (transformers para detección)

**DETR** (2020) reformula la detección como **predicción de un conjunto**:
1. Backbone CNN/ViT → *features* → **encoder transformer** (atención global sobre toda la imagen).
2. **Decoder** con N **object queries** (p. ej. 300 vectores aprendidos); cada query "pregunta" por un objeto
   y produce una caja + clase (o "no-objeto").
3. Entrenamiento con **emparejamiento húngaro** entre las N predicciones y los GT (coste = clase + L1 + GIoU):
   cada GT se asigna a **una sola** predicción → el modelo aprende a no duplicar → **no necesita NMS**.

Problemas del DETR original: converge lentísimo (500 épocas) y va mal en objetos pequeños.
Evoluciones:
- **Deformable DETR**: atención solo a unos pocos puntos de muestreo, multiescala → rápido y bueno en pequeños.
- **RT-DETR** (Baidu): *hybrid encoder* eficiente + selección de queries por IoU → **tiempo real**, compite con
  YOLO. Disponible en Ultralytics: `from ultralytics import RTDETR; RTDETR("rtdetr-l.pt")`.
- **D-FINE / DEIM**: refinan la regresión de cajas como distribuciones y aceleran la convergencia.
- **RF-DETR** (Roboflow, 2025): backbone **DINOv2** (ViT autosupervisado) + decoder deformable; SOTA en tiempo
  real y sobre todo **muy bueno haciendo fine-tuning** en datasets pequeños/de dominio (paquete `rfdetr`).

**¿YOLO o DETR?** YOLO: ecosistema maduro, muy rápido, fácil de exportar, multitarea. DETR moderno: sin NMS
(latencia más predecible, mejor en escenas con mucho solape), suele transferir mejor con backbone potente;
exportación a TensorRT algo más delicada. **Criterio: "lo decidiría midiendo ambos en nuestro dataset
y hardware objetivo (mAP vs ms/frame)"**.

## 9. Objetos pequeños (el balón, jugadores lejanos)

- Subir `imgsz` (entrenar e inferir a 1280).
- **Tiling / SAHI**: dividir la imagen en recortes solapados, detectar en cada uno y fusionar (NMS).
- Añadir cabeza **P2** (stride 4) → modelo `yolo11-p2.yaml`.
- Augmentations que no destruyan objetos pequeños (cuidado con `scale` y `mosaic` agresivos).
- Combinar con información temporal: interpolar la trayectoria del balón cuando falla la detección.

## 10. Análisis de errores de un detector

No te quedes en "mAP = 0.62". Lo que distingue a un buen candidato es **saber qué falla y por qué**:
- **FP**: fondo confundido (carteles con personas, reflejos), **duplicados** (NMS), localización pobre (IoU < umbral),
  confusión de clase (árbitro vs jugador).
- **FN**: objetos pequeños, **oclusión**, truncados en el borde, desenfoque de movimiento, poses raras, iluminación.
- Herramientas: matriz de confusión, mAP por tamaño (COCO: small < 32², medium, large > 96²), visualizar los
  peores casos (`val_batch*_pred.jpg`), ordenar imágenes por nº de FP/FN, **FiftyOne** para explorar.
- Acciones: más datos de los casos que fallan (*hard examples*), revisar etiquetas (¡muchos "errores" son
  errores de anotación!), cambiar umbrales, más resolución, augmentations específicas.

In [ ]:
# Mini análisis con nuestro pseudo-GT: ¿los FN son pequeños?
fn_h, tp_h = [], []
for k in gts:
    pk = np.array([p[2] for p in preds if p[0] == k and p[1] > 0.25]).reshape(-1, 4)
    if len(gts[k]) == 0: continue
    matched = box_iou(gts[k], pk).max(1) >= 0.5 if len(pk) else np.zeros(len(gts[k]), bool)
    h = gts[k][:, 3] - gts[k][:, 1]
    fn_h += list(h[~matched]); tp_h += list(h[matched])
plt.hist([tp_h, fn_h], bins=20, label=["detectadas (TP)", "perdidas (FN)"]); plt.xlabel("altura GT (px)"); plt.legend();
print(f"altura media TP = {np.mean(tp_h):.0f}px | FN = {np.mean(fn_h) if fn_h else float('nan'):.0f}px")